In [1]:
import polars as pl
import geopandas as gpd

In [2]:
df_dtb = pl.read_ods('RELATORIO_DTB_BRASIL_2025_MUNICIPIOS.ods')

In [3]:
df_dtb.head()

UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,Código Município Completo,Nome_Município
str,str,str,str,str,str,str,str,str
"""11""","""Rondônia""","""1102""","""Ji-Paraná""","""110005""","""Cacoal""","""00015""","""1100015""","""Alta Floresta D'Oeste"""
"""11""","""Rondônia""","""1102""","""Ji-Paraná""","""110005""","""Cacoal""","""00379""","""1100379""","""Alto Alegre dos Parecis"""
"""11""","""Rondônia""","""1101""","""Porto Velho""","""110002""","""Ariquemes""","""00403""","""1100403""","""Alto Paraíso"""
"""11""","""Rondônia""","""1102""","""Ji-Paraná""","""110004""","""Ji-Paraná""","""00346""","""1100346""","""Alvorada D'Oeste"""
"""11""","""Rondônia""","""1101""","""Porto Velho""","""110002""","""Ariquemes""","""00023""","""1100023""","""Ariquemes"""


In [4]:
df_dtb = (
    df_dtb.select(
        pl.col(['Nome_UF', 'Código Município Completo', 'Nome_Município'])
    )
    .rename({
        'Nome_UF': 'uf',
        'Código Município Completo': 'ibge',
        'Nome_Município': 'municipio'
    })
    .with_columns(
        pl.col('ibge').cast(pl.Int64) // 10
    )
)

In [5]:
df_dtb.head()

uf,ibge,municipio
str,i64,str
"""Rondônia""",110001,"""Alta Floresta D'Oeste"""
"""Rondônia""",110037,"""Alto Alegre dos Parecis"""
"""Rondônia""",110040,"""Alto Paraíso"""
"""Rondônia""",110034,"""Alvorada D'Oeste"""
"""Rondônia""",110002,"""Ariquemes"""


In [6]:
gdf_ufs = gpd.read_file('BR_UF_2025')

In [7]:
gdf_ufs.head()

,CD_UF,NM_UF,SIGLA_UF,CD_REGIAO,NM_REGIAO,SIGLA_RG,AREA_KM2,geometry
0,43,Rio Grande do Sul,RS,4,Sul,S,281707.150,"MULTIPOLYGON (((-53.52154 -33.25881, -53.51825..."
1,35,São Paulo,SP,3,Sudeste,SE,248219.485,"MULTIPOLYGON (((-48.03575 -25.35712, -48.03607..."
2,32,Espírito Santo,ES,3,Sudeste,SE,46074.440,"MULTIPOLYGON (((-40.88385 -21.16198, -40.88384..."
3,33,Rio de Janeiro,RJ,3,Sudeste,SE,43750.424,"MULTIPOLYGON (((-44.72025 -23.35934, -44.72029..."
4,41,Paraná,PR,4,Sul,S,199293.571,"MULTIPOLYGON (((-48.40723 -25.84254, -48.40732..."


In [8]:
gdf_ufs['NM_REGIAO'].unique()

<ArrowStringArray>
['Sul', 'Sudeste', 'Nordeste', 'Centro-oeste', 'Norte']
Length: 5, dtype: str

In [9]:
gdf_ufs = gdf_ufs.replace({
    'Centro-oeste': 'Centro-Oeste'
})

In [10]:
gdf_ufs['NM_REGIAO'].unique()

<ArrowStringArray>
['Sul', 'Sudeste', 'Nordeste', 'Centro-Oeste', 'Norte']
Length: 5, dtype: str

In [11]:
gdf_ufs.columns

Index(['CD_UF', 'NM_UF', 'SIGLA_UF', 'CD_REGIAO', 'NM_REGIAO', 'SIGLA_RG',
       'AREA_KM2', 'geometry'],
      dtype='str')

In [15]:
df_final = (
    gdf_ufs[['NM_UF', 'SIGLA_UF', 'NM_REGIAO']]
    .rename(columns={
        'NM_UF': 'uf',
        'SIGLA_UF': 'sigla_uf',
        'NM_REGIAO': 'regiao'
    })
    .merge(df_dtb.to_pandas(), on='uf')
)

In [16]:
df_final.head()

,uf,sigla_uf,regiao,ibge,municipio
0,Rio Grande do Sul,RS,Sul,430003,Aceguá
1,Rio Grande do Sul,RS,Sul,430005,Água Santa
2,Rio Grande do Sul,RS,Sul,430010,Agudo
3,Rio Grande do Sul,RS,Sul,430020,Ajuricaba
4,Rio Grande do Sul,RS,Sul,430030,Alecrim


In [17]:
df_final.to_csv('regioes_ufs_cidades_brasil.csv', index=False)